In [1]:
import pandas as pd
import numpy as np


In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier

In [3]:
df=pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [5]:
df.drop(columns=['PassengerId',"Cabin","Name","Ticket"],inplace=True)

In [6]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [8]:
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,1:],df.iloc[:,0:1],test_size=0.2,random_state=45)

In [9]:
(len(X_train)+len(X_test))==df.shape[0]

True

In [10]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [11]:
si_embarked=SimpleImputer(strategy="most_frequent")
si_age=SimpleImputer()
X_test_age=si_age.fit_transform(X_test[['Age']])
X_train_age=si_age.transform(X_train[['Age']])

X_test_embarked=si_embarked.fit_transform(X_test[["Embarked"]])
X_train_embarked=si_embarked.transform(X_train[["Embarked"]])

In [12]:
# one hot encoding on the columns embarked and sex
ohe_sex=OneHotEncoder(drop="first")
ohe_embarked=OneHotEncoder(drop="first")

ohe_sex.fit(X_train[["Sex"]])
ohe_embarked.fit(X_train_embarked)

X_train_sex=ohe_sex.transform(X_train[["Sex"]]).toarray()
X_test_sex=ohe_sex.transform(X_test[["Sex"]]).toarray()

X_train_embarked=ohe_embarked.transform(X_train_embarked).toarray()
X_test_embarked=ohe_embarked.transform(X_test_embarked).toarray()

In [13]:
columns=["Age"]+list(ohe_sex.get_feature_names_out())+list(ohe_embarked.get_feature_names_out())
columns

['Age', 'Sex_male', 'x0_Q', 'x0_S']

In [14]:
len(X_train_embarked)

712

In [15]:
X_train_arr=np.hstack((X_train_age,X_train_sex,X_train_embarked))
X_test_arr=np.hstack((X_test_age,X_test_sex,X_test_embarked))
len(X_test_arr)

179

In [16]:
X_train_temp=pd.DataFrame(X_train_arr,columns=columns)
X_test_temp=pd.DataFrame(X_test_arr,columns=columns)
X_train_temp

,Age,Sex_male,x0_Q,x0_S
0,38.000000,1.0,0.0,1.0
1,28.000000,1.0,0.0,1.0
2,9.000000,0.0,0.0,1.0
3,58.000000,0.0,0.0,1.0
4,34.000000,1.0,0.0,1.0
...,...,...,...,...
707,20.000000,1.0,0.0,1.0
708,27.000000,1.0,0.0,1.0
709,50.000000,1.0,0.0,0.0
710,28.763699,1.0,0.0,1.0


In [19]:
X_train_temp.index = X_train.index
X_test_temp.index = X_test.index

In [20]:
X_train=pd.concat([X_train[["Pclass","SibSp","Parch","Fare"]],X_train_temp],axis=1)
X_test=pd.concat([X_test[["Pclass","SibSp","Parch","Fare"]],X_test_temp],axis=1)
X_train.isnull().sum()




Pclass      0
SibSp       0
Parch       0
Fare        0
Age         0
Sex_male    0
x0_Q        0
x0_S        0
dtype: int64

In [21]:
print(X_train.index)
print(X_train_temp.index)

Index([332, 281, 147,  11, 405, 796, 510, 854, 253, 653,
       ...
        15, 632, 377, 580, 163, 725, 607, 544, 643, 414],
      dtype='int64', length=712)
Index([332, 281, 147,  11, 405, 796, 510, 854, 253, 653,
       ...
        15, 632, 377, 580, 163, 725, 607, 544, 643, 414],
      dtype='int64', length=712)


In [23]:
clf = DecisionTreeClassifier()
clf.fit(X_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [24]:
y_pred = clf.predict(X_test)
y_pred

array([0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1,
       0, 1, 1])

In [25]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.776536312849162